In [1]:
import os
import glob
import pandas as pd
import xlrd
import re
from pathlib import Path
import numpy as np
import xlsxwriter
import openpyxl
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import Alignment, Border, Side, PatternFill

In [ ]:
def get_first_row(worksheet, row_text_list):
    for text in row_text_list:
        text = text.lower()
        for row in worksheet.iter_rows():
            for cell in row:
                if cell.value and text in str(cell.value).lower():
                    return cell.row
    return -1

In [ ]:
def get_first_col(worksheet, col_text_list):
    for text in col_text_list:
        text = text.lower()
        for row in worksheet.iter_rows():
            for cell in row:
                if cell.value and text in str(cell.value).lower():
                    return cell.column
    return -1

In [ ]:
def correct_crop_name(incorrect_crop):
    with open("crop_correction.json", "r") as file:
        crop_correction = json.load(file)

    for correct, incorrect_list in crop_correction.items():
        for incorrect in incorrect_list:
                return correct
    
    return incorrect_crop

In [ ]:
def get_crop(ws, crop_row, col_idx, crop_type_row):
    # Global Variables
    kh = 'KHARIF'
    rb = 'RABI'
    kh_keywords1 = ('kar','kur','sam')
    rb_keywords1 = ('nav','rai')
    
    kh_keywords2 = ('K','sugar','can','maiz','ze','red','dgram')
    rb_keywords2 = ('R','®','hor','rse','segram','hour','green','engram','kgram','black')
    
    crop_raw = ws.cell(row=crop_row, column=col_idx).value
    if isinstance(crop_raw, str) and crop_raw is not None:
        crop_raw = crop_raw.strip().replace(' ','')
    else:
        crop_raw = ws.cell(row=crop_row, column=col_idx-1).value
        if isinstance(crop_raw, str) and crop_raw is not None:
            crop_raw = crop_raw.strip().replace(' ','')
        else:
            crop_raw = ws.cell(row=crop_row, column=col_idx-2).value
            if isinstance(crop_raw, str) and crop_raw is not None:
                crop_raw = crop_raw.strip().replace(' ','')
            else:
                crop_raw = None
    
    # set season
    if crop_raw is not None:
        crop_raw = re.sub(r'\s+', '', crop_raw)
        crop_raw = re.sub(r'UI', '', crop_raw)
        crop_raw = re.sub(r'IR', '', crop_raw)
        
        crop_type = ws.cell(row=crop_type_row, column=col_idx).value
        if isinstance(crop_type, str) and crop_type is not None:
            crop_type = crop_type.strip()
    
        if crop_type is not None and any(k in crop_type.lower() for k in kh_keywords1):
            season = kh
        elif crop_type is not None and any(r in crop_type.lower() for r in rb_keywords1):
            season = rb
        elif 'K' in crop_raw or any(k in crop_raw.lower() for k in kh_keywords2):
            season = kh
        elif 'R' in crop_raw or any(r in crop_raw.lower() for r in rb_keywords2):
            season = rb
        else:
            season = '----'
    
        if crop_type == 'I':
            crop_type = 'IR'
        elif crop_type.lower() == 'ui':
            crop_type = 'UI'
    
        # set crop names correctly
        crop = correct_crop_name(crop_raw)
    
        if crop == 'JOWAR':
            temp = crop_raw.lower().replace('jowar','')
            if 'k' in temp:
                season = kh
            elif any(r in temp for r in ('r','®')):
                season = rb
            else:
                season = '--'
    
        if 'PADDY' in crop:
            crop_type = '--'

        if crop_type in ('IR', 'UI'):
            crop = crop + '_' + crop_type

    return crop

In [3]:
def get_SL_to_WP(directory, st):
    #df to use
    df_state = pd.read_excel("ML_Template.xlsx", sheet_name="State", dtype=str)
    state = df_state['STATE'].iloc[0]
    year = df_state['YEAR'].iloc[0]
    season = df_state['SEASONCODE'].iloc[0]
    
    df_seasons = pd.read_excel("ML_Template.xlsx", sheet_name="Seasons", dtype=str)
    df_seasons = df_seasons[df_seasons['SEASONCODE']==season]
    
    df_samples = pd.read_excel("ML_Template.xlsx", sheet_name="Samples", dtype=str)
    df_districts = pd.read_excel("ML_Template.xlsx", sheet_name="Districts", dtype=str)
    df_crops = pd.read_excel("ML_Template.xlsx", sheet_name="Crops", dtype=str)
    df_crops = df_crops[df_crops['SEASONCODE']==season]

    #df to create
    df_Vill = pd.DataFrame(columns=['STATE','SEASONCODE','SAMPLE','DISTRICT',
                                    'CROPCODE','TALUKA','CIRCLE','VILLAGE'])
    
    df_ML = pd.DataFrame(columns=['YEAR','SEASONCODE','SEASONNAME','HSEASONNAME',
                                  'SAMPLE','SAMPLENAME','HSAMPLENAME','STATE',
                                  'STATENAME','SHORTSTATE','HSTATENAME','ROCODE',
                                  'RONAME','HRONAME','SROCODE','SRONAME','HSRONAME',
                                  'DISTRICT','DISTRICTNAME','HDISTRICTNAME',
                                  'SELORDER','EXPT','CROPCODE','CROPNAME',
                                  'HCROPNAME','STATUS','EXPTID',])
    
    # Get all .xlsx and .xls files in the directory
    excel_files = glob.glob(os.path.join(directory, "*.xlsx"))
    
    for file_path in excel_files:
        file_name = os.path.basename(file_path)
        distt = Path(file_name).stem
        # Determine file type and use appropriate library
        if file_path.endswith('.xlsx'):
            try:
                wb = load_workbook(file_path, data_only=True)
                for sheet_name in wb.sheetnames:
                    if "cent" in sheet_name.lower() or "sta" in sheet_name.lower():
                        sample = '1' if "cent" in sheet_name.lower() else '2'
                        
                        ws = wb[sheet_name]
                        
                        # setting which cells to scan
                        row_text_list = ['block', 'name of the village', 'name of village']
                        start_row = 0
                        title_row = 0
                        while start_row <= 0:
                            title_row = get_first_row(ws, row_text_list)
                            crop_row = title_row
                            crop_type_row = title_row + 1
                            os_row = title_row + 2
                            start_row = title_row + 3
                            
                        col_text_list = ['Expts. & O.S', 'Expts', 'Expt', 'O.S']
                        start_col = 0
                        while start_col <= 0:
                            start_col = get_first_col(ws, col_text_list)
                            block_col = start_col - 2
                            village_col = start_col - 1
                        
                        end_row = ws.max_row + 1
                        end_col = ws.max_column + 1

                        for col_idx in range(start_col, end_col):
                            # get correct crop name and type
                            crop1 = get_crop(ws, crop_row, col_idx, crop_type_row)
                            crop = crop1
                            
                            vill_count = 0
                            for row_idx in range(start_row, end_row):
                                cell_val = ws.cell(row=row_idx, column=col_idx).value
                                if isinstance(cell_val, str) and cell_val is not None:
                                    cell_val = cell_val.strip()
                                
                                if cell_val is not None:
                                    if block_col>0 and village_col>0:
                                        block = ws.cell(row=row_idx, column=block_col).value
                                        village = ws.cell(row=row_idx, column=village_col).value

                                        if isinstance(block, str) and block is not None:
                                            if isinstance(village, str) and village is not None:
                                                #if block is not None and village is not None:
                                                #set crop code 4 digits
                                                if '@' in str(cell_val):
                                                    crop = crop1 + '_A'
                                                    cell_val = cell_val.strip().replace('@','')
                                                elif '#' in str(cell_val):
                                                    crop = crop1 + '_B'
                                                    cell_val = cell_val.strip().replace('#','')

                                                crop_code = df_crops[df_crops['CROPNAME']==crop]['CROPCODE'].iloc[0]


                                                #########################################################################################
                                                m_digit  = re.search(r"\d+", str(cell_val))
                                                
                                                if m_digit is not None:
                                                    selorder = m_digit.group()
                                                    selorder = str(selorder)

                                                    if len(selorder) == 1:
                                                        selorder = '0' + selorder
                                                    
                                                    if str(block) is not None:
                                                        block = block.upper()
                                                    if str(circle) is not None:
                                                        circle = circle.upper()
                                                    if str(village) is not None:
                                                        village = village.upper()
                                                    
                                                    df_village_row = {
                                                        'STATE': state,
                                                        'SEASONCODE': season,
                                                        'SAMPLE': sample,
                                                        'DISTRICT': district_code,
                                                        'DISTRICTNAME': district_name,
                                                        'CROPCODE': crop_code,
                                                        'block': block,
                                                        'CIRCLE': circle,
                                                        'VILLAGE': village
                                                    }
                                                    df_Vill = pd.concat([df_Vill, pd.DataFrame([df_village_row])], 
                                                                        ignore_index=True)
                                                    
                                                    df_ML_row = {
                                                        'YEAR': [year, year],
                                                        'STATE': [state, state],
                                                        'SEASONCODE': [season, season],
                                                        'SAMPLE': [sample, sample],
                                                        'DISTRICT': [district_code, district_code],
                                                        'DISTRICTNAME': [district_name, district_name],
                                                        'CROPCODE': [crop_code, crop_code],
                                                        'SELORDER': [selorder, selorder],
                                                        'EXPT': ['1','2']
                                                    }
                                                    
                                                    df_ML = pd.concat([df_ML, pd.DataFrame(df_ML_row, dtype=str)], ignore_index=True)

            except Exception as e:
                print(f"Error reading {file_name}, {st[0]}, {incorrect_crop}, {crop_type}, {season}, {distt}, {plan} (openpyxl): {e}")
    return df_pivot

In [5]:
if __name__ == "__main__":
    curr_dir = Path.cwd()
    directory = curr_dir / "SL 2.0"
    excel_path = curr_dir / "SL_to_WP.xlsx"
    df_Vill, df_ML = get_SL_to_ML(directory)

    with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
        df_Vill.to_excel(writer, sheet_name="Villages", index=False)
        df_ML.to_excel(writer, sheet_name="ML", index=False)